# Scoring benchmark

Score `P(Y = y | x)`, the probability of the pattern each held-out row actually has, from one fitted
model. Data are synthetic: 6 features, linear margins, equicorrelated Σ at ρ = 0.4, 3,000 training rows,
100 scored rows per d. The model is `MultivariateProbit(inner="linear", dependence="pairwise")`.

Every column scores that same fitted model (η̂, Σ̂), except statsmodels, which fits one `Probit` per
outcome and has no Σ, so its pattern probability is a product of marginals.

**Reference:** `scipy.stats.multivariate_normal.cdf` with `abseps=1e-7, releps=1e-6`, one row at a time.
**Error:** absolute difference from the reference, median and max over rows. **Time:** wall-clock per row,
after one warm-up call; statsmodels is timed on prediction only.

The logic is in `bench.py`. To include the keyed orthant evaluator at `resolution="high"`, put a key in
`$ORTHANT_KEY` or `~/.orthant/key` and rerun; without one it runs only at d ≤ 3 and `resolution="low"`.
R `mvProbit` is not run here.

In [1]:
import warnings, platform, os
import pandas as pd
import bench
warnings.filterwarnings("ignore")
print("orthant keyed:", bench.ORTHANT_KEYED, "| pybhatlib:", bench.mvncd_batch is not None,
      "| statsmodels:", bench.sm is not None)
print(platform.processor() or platform.machine(), os.cpu_count(), "cores")

orthant keyed: True | pybhatlib: True | statsmodels: True
x86_64 2 cores


In [2]:
rows, median_p = [], {}
for d in (3, 5, 8, 12, 16):
    r, median_p[d] = bench.run_cell(d)
    rows += r
df = pd.DataFrame(rows)
df.to_csv("results.csv", index=False)
median_p

{3: 0.3479461366480382,
 5: 0.1539021319736073,
 8: 0.10058867730376865,
 12: 0.022052493788054097,
 16: 0.0024369471356400427}

In [3]:
def cell(r):
    t = f"{r.s_per_row:.1e} s"
    return t if r.method == "reference" else f"{t} / {r.median_err:.4f} ({r.max_err:.4f})"

table = df.assign(cell=df.apply(cell, axis=1)).pivot(index="d", columns="method", values="cell").fillna("—")
table

method,SciPy default,mvp orthant (high),mvp quadrature,pybhatlib ovus,reference,statsmodels
d,,,,,,
3,4.2e-03 s / 0.0000 (0.0000),8.9e-06 s / 0.0001 (0.0002),6.0e-05 s / 0.0000 (0.0001),6.4e-05 s / 0.0001 (0.0009),2.4e-01 s,4.4e-07 s / 0.0284 (0.0989)
5,7.1e-03 s / 0.0000 (0.0000),1.6e-05 s / 0.0001 (0.0003),3.4e-02 s / 0.0000 (0.0001),9.4e-05 s / 0.0004 (0.0019),5.8e-01 s,5.8e-07 s / 0.0245 (0.1288)
8,1.3e-02 s / 0.0000 (0.0000),2.3e-05 s / 0.0001 (0.0005),—,1.4e-04 s / 0.0003 (0.0023),1.4e+00 s,8.4e-07 s / 0.0144 (0.1152)
12,1.9e-02 s / 0.0000 (0.0000),3.7e-05 s / 0.0001 (0.0007),—,2.0e-04 s / 0.0002 (0.0035),1.7e+00 s,1.1e-06 s / 0.0064 (0.0782)
16,2.6e-02 s / 0.0000 (0.0000),6.0e-05 s / 0.0000 (0.0006),—,2.5e-04 s / 0.0000 (0.0012),1.1e+00 s,1.7e-06 s / 0.0015 (0.0534)
